# NBA 3.1 HCP Universe — Validation & Hotspot Analysis

This notebook validates the constructed universe against baselines and profiles missed writers.

**Sections:**
1. NBA 3.0 overlap analysis
2. Kerendia writer validation metrics (M1–M4)
3. Hotspot analysis: profiling writers missed by C1
4. Greedy set addition: optimal combination strategy
5. Call history overlap (C5 exploration)

**Prerequisite:** Run `02_Universe_Construction` first — this notebook expects `universe_flagged`, `c1_core`, `c2_writers`, etc.

In [0]:
%run "/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
# ============================================================
# Imports & Parameters
# ============================================================
from functools import reduce
from pyspark.sql import functions as F, Window

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"

LAAD_V6_TBL = f"`{CATALOG}`.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v6"
NBA30_PATH = (
    f"/Volumes/{CATALOG}/"
    "kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
    "dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)
MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"

print("Parameters loaded.")

In [0]:
# ============================================================
# Rebuild all universe components (self-contained)
# ============================================================
import builtins
from pyspark.sql import functions as F, Window

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SOURCE_TABLE = f"`{CATALOG}`.pace_ker_in_house_schema.lrx_nbrx"
KERENDIA_NBRX_PATH = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/kerendia_nbrx_from_raw_fct_rx"
)
MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"
LOOKBACK_START = "2024-10-01"
LOOKBACK_END = "2026-09-30"
INDICATION_COL = "INDICATION_GROUP_APP1"
PRODUCT_COL = "PRODUCT_CLASS"
METRIC_COL = "NBRX"
INDICATIONS = ["T2D", "CKD", "HF", "T1D"]
DRUG_POOLS = {"SGLT2_GLP1": ["SGLT2", "GLP1"], "ACE_ARB": ["ACE", "ARB"]}

# Load claims lookback
df_src = spark.table(SOURCE_TABLE)
lrx = df_src.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)

# Decile function
def compute_deciles(df, indication, pool_products):
    filtered = df.filter(
        (F.col(INDICATION_COL).contains(indication)) &
        (F.col(PRODUCT_COL).isin(pool_products))
    )
    hcp_vol = filtered.groupBy("PROVIDER_ID").agg(F.sum(METRIC_COL).alias("TOTAL_NBRX"))
    total_vol = hcp_vol.agg(F.sum("TOTAL_NBRX")).collect()[0][0]
    if total_vol is None or total_vol == 0:
        return None, 0, 0
    vol_per_decile = float(total_vol) / 10.0
    w = Window.orderBy(F.asc("TOTAL_NBRX"), F.asc("PROVIDER_ID"))
    hcp_vol = hcp_vol.withColumn("CUM_NBRX", F.sum("TOTAL_NBRX").over(w))
    hcp_vol = hcp_vol.withColumn(
        "DECILE", F.least((F.col("CUM_NBRX") / F.lit(vol_per_decile)).cast("int") + 1, F.lit(10))
    ).drop("CUM_NBRX")
    return hcp_vol, hcp_vol.count(), int(total_vol)

# Compute all 8 decile sets
all_decile_dfs = {}
for ind in INDICATIONS:
    for pool_name, pool_products in DRUG_POOLS.items():
        decile_df, n_hcps, total_vol = compute_deciles(lrx, ind, pool_products)
        if decile_df is not None:
            all_decile_dfs[(ind, pool_name)] = decile_df

set1 = all_decile_dfs[("T2D", "SGLT2_GLP1")]
set3 = all_decile_dfs[("CKD", "SGLT2_GLP1")]

# C1 Core
c1_core = (
    set1.filter(F.col("DECILE") >= 2).select("PROVIDER_ID")
    .unionByName(set3.filter(F.col("DECILE") >= 2).select("PROVIDER_ID"))
    .distinct()
)
c1_ct = c1_core.count()

# C2 Kerendia Writers
ker_fct_rx = spark.read.format("delta").load(KERENDIA_NBRX_PATH)
ker_fct_rx = ker_fct_rx.withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))
c2_writers = ker_fct_rx.filter(F.col("NBRX_KERENDIA_OVERALL") > 0).select("PROVIDER_ID").distinct()
c2_ct = c2_writers.count()

# C3 Specialists
specialty_hcps = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct()
neph_c3 = specialty_hcps.filter(F.col("SPECIALTY_GROUP") == "NEPH").select("PROVIDER_ID").distinct()
card_c3 = specialty_hcps.filter(F.col("SPECIALTY_GROUP") == "CARD").select("PROVIDER_ID").distinct()
c3_specialists = neph_c3.unionByName(card_c3).distinct()

# C4 Targeting
query_c4 = """SELECT DISTINCT BAYER_ID as BH_ID FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
    WHERE FF IN ('CR1','CR2','CR4','CR5')
    AND ((UPPER(PRODUCT_1)='KERENDIA') OR (UPPER(PRODUCT_2)='KERENDIA'))"""
try:
    c4_raw = Get_Data_Snowflakes(query_c4)
    id_map = spark.table(MAPPING_TABLE).select(
        F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"),
        F.col("BH_ID").cast("string").alias("BH_ID")
    )
    c4_mapped = c4_raw.select(F.col("BH_ID").cast("string").alias("BH_ID")).distinct().join(id_map, on="BH_ID", how="inner").select("PROVIDER_ID").distinct()
except:
    c4_mapped = spark.createDataFrame([], "PROVIDER_ID: string")

# Combined universe
universe = c1_core.unionByName(c2_writers).unionByName(c3_specialists).unionByName(c4_mapped).distinct()
universe_ct = universe.count()

# Flagged universe
universe_flagged = (
    universe
    .join(c1_core.withColumn("C1_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .join(c2_writers.withColumn("C2_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .join(c3_specialists.withColumn("C3_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .join(c4_mapped.withColumn("C4_FLAG", F.lit(1)), on="PROVIDER_ID", how="left")
    .fillna(0, subset=["C1_FLAG", "C2_FLAG", "C3_FLAG", "C4_FLAG"])
)
spec_dedup = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct().groupBy("PROVIDER_ID").agg(
    F.first("SPECIALTY_GROUP", ignorenulls=True).alias("SPECIALTY_GROUP")
)
universe_flagged = universe_flagged.join(spec_dedup, on="PROVIDER_ID", how="left")

print(f"Universe rebuilt: {universe_ct:,} HCPs")
print(f"  C1: {c1_ct:,} | C2: {c2_ct:,} | C3: {c3_specialists.count():,} | C4: {c4_mapped.count():,}")

In [0]:
# ============================================================
# Load NBA 3.0 universe baseline
# ============================================================
nba30 = spark.read.format("delta").load(NBA30_PATH)
nba30_ct = nba30.count()
nba30_ids = nba30.select("PROVIDER_ID")

print(f"NBA 3.0 Universe: {nba30_ct:,} HCPs")
for flag in ["KERENDIA_FLAG", "SGLT2_GLP1_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]:
    if flag in nba30.columns:
        ct = nba30.filter(F.col(flag) == 1).count()
        print(f"  {flag} = 1: {ct:,} ({100*ct/nba30_ct:.1f}%)")

In [0]:
# ============================================================
# Universe overlap vs NBA 3.0 baseline
# ============================================================
universe = universe_flagged.select("PROVIDER_ID").distinct()
universe_ct = universe.count()

overlap = universe.join(nba30_ids, on="PROVIDER_ID", how="inner")
overl_ct = overlap.count()
univ_only = universe.join(nba30_ids, on="PROVIDER_ID", how="left_anti")
univ_only_ct = univ_only.count()
nba30_only = nba30_ids.join(universe, on="PROVIDER_ID", how="left_anti")
nba30_only_ct = nba30_only.count()

print(f"Universe:    {universe_ct:,}")
print(f"NBA 3.0:     {nba30_ct:,}")
print(f"Overlap:     {overl_ct:,} ({100*overl_ct/nba30_ct:.1f}% of NBA 3.0)")
print(f"Universe-only: {univ_only_ct:,}")
print(f"NBA30-only:    {nba30_only_ct:,}")

In [0]:
# ============================================================
# Validation metrics
# M1: NBA 3.0 KERENDIA_FLAG
# M2: lrx_nbrx paid NBRX>0 (24-mo) -- approximated via prior notebook if available
# M3: LAAD V6 TRX>0 (24-mo)
# M4: FCT_RX paid TRX (24-mo)
# ============================================================
laad_v6 = spark.table(LAAD_V6_TBL)

# M1
m1_writers = nba30.filter(F.col("KERENDIA_FLAG") == 1).select("PROVIDER_ID").distinct()
m1_ct = m1_writers.count()

# M2
m2_writers = laad_v6.filter(
    (F.col("WEEK_ENDING_DATE") >= "20241001") &
    (F.col("WEEK_ENDING_DATE") <= "20260930") &
    (F.col("PRODUCT_GROUP") == "KERENDIA") &
    (F.col("NBRX") > 0)
).select("PROVIDER_ID").distinct()
m2_ct = m2_writers.count()

# M3
m3_writers = laad_v6.filter(
    (F.col("WEEK_ENDING_DATE") >= "20241001") &
    (F.col("WEEK_ENDING_DATE") <= "20260930") &
    (F.col("PRODUCT_GROUP") == "KERENDIA") &
    (F.col("TRX") > 0)
).select("PROVIDER_ID").distinct()
m3_ct = m3_writers.count()

# M4
query_m4 = f'''
    SELECT DISTINCT A.PROVIDER_ID
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE A.CLAIM_TYPE = 'PD'
    AND A.SVC_DT >= '{LOOKBACK_START}'
    AND A.SVC_DT <= '{LOOKBACK_END}'
    AND UPPER(TRIM(B.PRODUCT_GROUP)) = 'KERENDIA'
'''
try:
    m4_raw = Get_Data_Snowflakes(query_m4)
    m4_writers = m4_raw.select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID")).distinct()
    m4_ct = m4_writers.count()
except Exception as e:
    print(f"WARNING: M4 Snowflake query failed: {e}")
    m4_writers = spark.createDataFrame([], "PROVIDER_ID: string")
    m4_ct = 0

print(f"Baselines: M1={m1_ct:,} | M2={m2_ct:,} | M3={m3_ct:,} | M4={m4_ct:,} | C2={c2_writers.count():,}")

In [0]:
# ============================================================
# Capture of M1-M4 and C2 within the final universe
# ============================================================
metrics = [
    ("M1: NBA30 KERENDIA_FLAG", m1_writers, m1_ct),
    ("M2: LAAD V6 NBRX>0", m2_writers, m2_ct),
    ("M3: LAAD V6 TRX>0", m3_writers, m3_ct),
    ("M4: FCT_RX paid TRX", m4_writers, m4_ct),
    ("C2: FCT_RX NBRX overall", c2_writers, c2_writers.count()),
]

print(f"{'Metric':<28} {'Baseline':>10} {'Captured':>10} {'%':>8} {'Missed':>10}")
print("-" * 75)
for label, df_metric, ct in metrics:
    if ct == 0:
        print(f"{label:<28} {ct:>10,} {0:>10,} {'N/A':>8} {0:>10,}")
        continue
    captured = df_metric.join(universe, on="PROVIDER_ID", how="inner").count()
    missed = ct - captured
    pct = 100*captured/ct
    print(f"{label:<28} {ct:>10,} {captured:>10,} {pct:>7.1f}% {missed:>10,}")

In [0]:
# ============================================================
# Kerendia writers NOT captured by C1 core alone
# ============================================================
missed_by_c1 = c2_writers.join(c1_core, on="PROVIDER_ID", how="left_anti")
missed_ct = missed_by_c1.count()
c2_ct = c2_writers.count()

print(f"Kerendia writers missed by C1 core: {missed_ct:,} / {c2_ct:,} ({100*missed_ct/c2_ct:.1f}%)")

# Recovery path via C3 / C4
missed_c3 = missed_by_c1.join(c3_specialists, on="PROVIDER_ID", how="inner").count()
missed_c4 = missed_by_c1.join(c4_mapped, on="PROVIDER_ID", how="inner").count()
print(f"  Recovered by C3 (specialty): {missed_c3:,}")
print(f"  Recovered by C4 (targeting): {missed_c4:,}")
print(f"  Recovered by C2: {missed_ct:,} (by definition)")

In [0]:
# ============================================================
# Specialty profile of writers missed by C1
# ============================================================
specialty_profile = (
    missed_by_c1.join(
        universe_flagged.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct(),
        on="PROVIDER_ID", how="left"
    )
    .groupBy("SPECIALTY_GROUP")
    .count()
    .orderBy(F.desc("count"))
)
display(specialty_profile)

In [0]:
# ============================================================
# Kerendia NBRX/TRX profile of writers missed by C1
# ============================================================
missed_volume = (
    missed_by_c1.join(
        ker_fct_rx.select("PROVIDER_ID", "NBRX_KERENDIA_OVERALL", "TRX_KERENDIA_OVERALL"),
        on="PROVIDER_ID", how="left"
    )
)

# NBRX bucket distribution
missed_volume = missed_volume.withColumn(
    "NBRX_BUCKET",
    F.when(F.col("NBRX_KERENDIA_OVERALL") == 1, "1")
    .when(F.col("NBRX_KERENDIA_OVERALL").between(2, 3), "2-3")
    .when(F.col("NBRX_KERENDIA_OVERALL").between(4, 5), "4-5")
    .when(F.col("NBRX_KERENDIA_OVERALL").between(6, 10), "6-10")
    .otherwise("11+")
)

display(missed_volume.groupBy("NBRX_BUCKET").count().orderBy("NBRX_BUCKET"))
print("Summary stats:")
missed_volume.describe("NBRX_KERENDIA_OVERALL", "TRX_KERENDIA_OVERALL").show()

In [0]:
# ============================================================
# Greedy strategy: start with Core (Set1 ∪ Set3), then add
# sets in order of writer-recovery-per-HCP ratio.
# ============================================================
print("GREEDY SET ADDITION (starting from Core = Set1 ∪ Set3)")
print("=" * 95)

set_candidates = [
    ("Set2 T2D×ACE/ARB", all_decile_dfs[("T2D", "ACE_ARB")].filter(F.col("DECILE") >= 2).select("PROVIDER_ID")),
    ("Set4 CKD×ACE/ARB", all_decile_dfs[("CKD", "ACE_ARB")].filter(F.col("DECILE") >= 2).select("PROVIDER_ID")),
    ("Set5 HF×SGLT2/GLP1", all_decile_dfs[("HF", "SGLT2_GLP1")].filter(F.col("DECILE") >= 2).select("PROVIDER_ID")),
    ("Set6 HF×ACE/ARB", all_decile_dfs[("HF", "ACE_ARB")].filter(F.col("DECILE") >= 2).select("PROVIDER_ID")),
    ("Set7 T1D×SGLT2/GLP1", all_decile_dfs[("T1D", "SGLT2_GLP1")].filter(F.col("DECILE") >= 2).select("PROVIDER_ID")),
    ("Set8 T1D×ACE/ARB", all_decile_dfs[("T1D", "ACE_ARB")].filter(F.col("DECILE") >= 2).select("PROVIDER_ID")),
]

current = c1_core
current_ct = current.count()
current_writers = c2_writers.join(current, on="PROVIDER_ID", how="inner").count()

print(f"{'Step':<5} {'Action':<25} {'Universe':>10} {'Ker Writers':>12} {'%Captured':>10} {'Marg HCPs':>12} {'Marg Ker':>10}")
print("-" * 100)
print(f"{0:<5} {'Core (Set1∪Set3)':<25} {current_ct:>10,} {current_writers:>12,} {100*current_writers/c2_ct:>9.1f}% {'-':>12} {'-':>10}")

step = 1
remaining = set_candidates.copy()
while remaining:
    best = None
    best_ratio = -1
    for label, df_set in remaining:
        new_univ = current.unionByName(df_set).distinct()
        marg_hcps = new_univ.count() - current_ct
        new_writers = c2_writers.join(new_univ, on="PROVIDER_ID", how="inner").count()
        marg_writers = new_writers - current_writers
        ratio = marg_writers / marg_hcps if marg_hcps > 0 else 0
        if ratio > best_ratio:
            best_ratio = ratio
            best = (label, df_set, new_univ, marg_hcps, new_writers, marg_writers)
    label, df_set, new_univ, marg_hcps, new_writers, marg_writers = best
    current = new_univ
    prev_ct = current_ct
    current_ct = current.count()
    current_writers = new_writers
    print(f"{step:<5} {label:<25} {current_ct:>10,} {current_writers:>12,} {100*current_writers/c2_ct:>9.1f}% {marg_hcps:>12,} {marg_writers:>10,}")
    remaining = [(l, d) for (l, d) in remaining if l != label]
    step += 1

In [0]:
# ============================================================
# C5: Call History Overlap
# Uses Python snowflake.connector (from 00_connection) to query
# CPH_DB_PROD directly, bypassing Spark Snowflake connector issues
# ============================================================
import snowflake.connector
import pandas as pd

# Correct columns: BAYER_CUST_ID (=BH_ID), SOURCE_CUST_ID (join key)
query_c5 = """
    SELECT DISTINCT B.BAYER_CUST_ID AS BH_ID
    FROM MODEL_V2.FACT_CALL A
    INNER JOIN MODEL_V2.MAP_CUSTOMER_XREF B
        ON A.SOURCE_CUST_ID = B.SOURCE_CUST_ID
    WHERE A.STAT = 'Submitted_vod'
      AND UPPER(A.DETL_PROD) LIKE '%KERENDIA%'
      AND B.BAYER_CUST_ID NOT LIKE '%BHO%'
      AND A.CALL_DATE >= '2023-01-01'
"""

try:
    conn = snowflake.connector.connect(
        user=snowflake_user,
        account=snowflake_account,
        private_key=p_key,
        role="BAY_CPH_CDP_PROD_CR_US_EAST_1_MVXPF_MVXPF",
        warehouse="PROD_CYRUS_BI_WH",
        database="CPH_DB_PROD",
        schema="MODEL_V2"
    )
    cur = conn.cursor()
    cur.execute(query_c5)
    c5_pdf = cur.fetch_pandas_all()
    print(f"C5 query returned {len(c5_pdf)} rows")
    cur.close()
    conn.close()

    if len(c5_pdf) == 0:
        print("WARNING: C5 query returned no results.")
        raise ValueError("Empty C5 result")
    c5_bh = spark.createDataFrame(c5_pdf).select(F.col("BH_ID").cast("string").alias("BH_ID")).distinct()
    id_map = spark.table(MAPPING_TABLE).select(
        F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"),
        F.col("BH_ID").cast("string").alias("BH_ID")
    )
    c5_hcps = c5_bh.join(id_map, on="BH_ID", how="inner").select("PROVIDER_ID").distinct()
    c5_ct = c5_hcps.count()
    c5_in = c5_hcps.join(universe, on="PROVIDER_ID", how="inner").count()
    c5_out = c5_hcps.join(universe, on="PROVIDER_ID", how="left_anti").count()

    print(f"Called HCPs since 2023: {c5_ct:,}")
    print(f"In universe:            {c5_in:,} ({100*c5_in/c5_ct:.1f}%)")
    print(f"Outside universe:       {c5_out:,} ({100*c5_out/c5_ct:.1f}%)")

    # Profile outside HCPs
    c5_outside = c5_hcps.join(universe, on="PROVIDER_ID", how="left_anti")
    c5_out_spec = c5_outside.join(
        lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct().groupBy("PROVIDER_ID").agg(
            F.first("SPECIALTY_GROUP", ignorenulls=True).alias("SPECIALTY_GROUP")
        ),
        on="PROVIDER_ID", how="left"
    )
    print(f"\nOutside HCPs by specialty:")
    display(c5_out_spec.groupBy("SPECIALTY_GROUP").count().orderBy(F.desc("count")))
except Exception as e:
    print(f"WARNING: C5 exploration failed: {e}")

## Validation Summary

Key validations performed:

* **NBA 3.0 overlap** — measures continuity vs prior universe
* **M1–M4 metrics** — independent Kerendia writer / TRX coverage checks
* **Hotspot analysis** — profiles C2 writers missed by C1
* **Greedy set addition** — quantifies marginal writer capture from non-core sets
* **C5 exploration** — estimates how many called HCPs sit outside the current universe

**Expected pattern:** Core D2+ should capture ~80% of Kerendia writers; full universe should recover 100% through C2 and stay under 300K.

Following your preferences, I used your existing V6 methodology as inspiration but kept the notebook modular and easier to adapt.